# 🧭 Self-Discover: Composing Task-Specific Reasoning Structures

## Learning Objectives
In this notebook, you will learn:
1. **The Self-Discover pattern** - Separate "how to reason about this type of task" from "solve this instance"
2. **SELECT, ADAPT, IMPLEMENT** - Turn a generic catalog of reasoning modules into a reusable, typed reasoning plan
3. **SOLVE** - Follow the discovered plan on a specific instance
4. **Measuring the difference** - Check direct answers and Self-Discover answers against a brute-force solution, over several samples

## Where this fits
`04_chain_of_verification.ipynb` improved an answer *after* it was written. Self-Discover spends its extra calls *before* the answer: it first works out what a good reasoning process for this kind of task looks like. `06_constitutional_ai_critique_loop.ipynb` returns to after-the-fact critique, with named rules.

## Prerequisites
- `with_structured_output` and `ChatPromptTemplate` - `modules/04_langchain_agents_and_middleware/notebooks/03_structured_output.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📖 Part 1: The Self-Discover Pattern

Most agent patterns either answer a task directly, or answer-then-critique-and-revise (Reflection/Reflexion). **Self-Discover** takes a different approach: before attempting the task at all, the agent composes a **custom, step-by-step reasoning plan** tailored to the *type* of problem it's facing, and only then applies that plan to the specific instance.

The pattern (from Zhou et al., "SELF-DISCOVER: Large Language Models Self-Compose Reasoning Structures") breaks this into **four distinct stages**:

1. **SELECT** — From a fixed catalog of generic reasoning modules (e.g. "Critical Thinking", "Break down into sub-problems"), pick the subset relevant to this task.
2. **ADAPT** — Rephrase each selected generic module into a task-specific instruction.
3. **IMPLEMENT** — Compose the adapted instructions into a concrete, ordered, structured reasoning plan — the *reusable* "discovered structure" for this **task type**, not yet an answer to any specific instance.
4. **SOLVE** — Follow the discovered structure to actually solve the specific task instance and produce the final answer.

The key insight: stages 1–3 only look at the *shape* of the task (its type), while stage 4 is the only stage that touches the *specific* instance. In principle, the same discovered structure from stages 1–3 could be reused to solve many different instances of the same task type.

| Property | Value |
|---|---|
| Origin | Zhou et al. (Google DeepMind), *Self-Discover: Large Language Models Self-Compose Reasoning Structures* (2024). [arXiv:2402.03620](https://arxiv.org/abs/2402.03620) |

---
## 📦 Step 1: Environment Setup

Load the API key and create two models: a deterministic one for the SELECT/ADAPT/IMPLEMENT stages, and one with some randomness for the answers we sample in Part 4.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: Deterministic stages + a sampling model for the answers
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")                            # SELECT / ADAPT / IMPLEMENT
sampling_llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)  # answers sampled in Part 4

print(f"✅ Stages: {llm.model_name} (temperature={llm.temperature}); answers: temperature={sampling_llm.temperature}")

---
## 🧩 Part 2: The Catalog and the Task

### 2.1 The Reasoning Module Catalog

Self-Discover starts from a **fixed, generic catalog** of reasoning modules - short descriptions of ways of thinking that are useful across many kinds of problems. The agent never invents new modules; it only *selects* and *adapts* from this catalog. This is what makes the discovered structure reusable and inspectable, instead of an opaque chain-of-thought.

In [ ]:
# ============================================================================
# REASONING MODULE CATALOG: 10 generic ways of thinking
# ============================================================================
REASONING_MODULES = [
    "Critical Thinking: Analyze the problem from multiple perspectives, question assumptions, "
    "and evaluate the evidence or reasoning at each step rather than accepting the first plausible answer.",

    "Decomposition: Break the problem down into smaller, more manageable sub-problems that can be "
    "addressed somewhat independently before combining their results.",

    "Use Analogies: Map the problem onto a similar, more familiar problem or scenario, and borrow "
    "the solution strategy that worked there.",

    "Step-by-Step Chronological Reasoning: Process the problem's information in a strict, ordered "
    "sequence, tracking how the state of the world evolves after each new piece of information.",

    "Consider Trade-offs: Weigh the pros and cons of multiple candidate solutions or interpretations "
    "before committing to one.",

    "Constraint Propagation: Identify the hardest, most restrictive constraints first, use them to "
    "eliminate impossible options immediately, and iteratively narrow the remaining solution space.",

    "Case Analysis: Enumerate the distinct possible cases or branches explicitly, test each one "
    "against every constraint, and discard any case that produces a contradiction.",

    "Working Backwards: Start from the desired end goal or the shape of a valid final answer, and "
    "reason backward about what conditions must have held earlier.",

    "Simplification: Restate the problem in the simplest possible terms, stripping away irrelevant "
    "details and flavor text, before attempting to solve it.",

    "Verification / Self-Check: After proposing an answer, re-check every stated constraint one by "
    "one against the proposed solution to confirm that none of them is violated.",
]

print(f"✅ Catalog loaded with {len(REASONING_MODULES)} modules:", [m.split(":")[0] for m in REASONING_MODULES])

### 2.2 The Task

A small but genuinely tricky **multi-constraint scheduling puzzle**, with a conditional clue (an "if... then...") and an ordering clue - the kind of thing a model answering "off the top of its head" gets wrong by dropping one constraint or mis-applying the conditional.

We never write the answer down: code **brute-forces** every assignment and keeps the ones that satisfy all four clues. That gives a ground truth to check every model answer against.

In [ ]:
# ============================================================================
# THE TASK: Type description, specific instance, and a brute-force solution
# ============================================================================
from itertools import permutations

TASK_TYPE_DESCRIPTION = (
    "A multi-constraint logic/scheduling puzzle: several entities must each be assigned to exactly "
    "one slot/category out of a small fixed set, subject to a handful of constraints that include "
    "negative constraints, relative-ordering constraints, and conditional (if-then) constraints. "
    "Exactly one assignment satisfies all constraints simultaneously."
)

TASK_INSTANCE = ("Three analysts -- Priya, Quinn, and Rae -- must each be assigned to exactly one of three "
    "interview time slots: 9:00 AM, 11:00 AM, and 2:00 PM (no two analysts share a slot).\n\n"
    "Clues:\n"
    "1. Priya is not in the 9:00 AM slot.\n"
    "2. Quinn's slot is earlier in the day than Rae's slot.\n"
    "3. If Priya is in the 2:00 PM slot, then Quinn is in the 9:00 AM slot.\n"
    "4. Rae is not in the 11:00 AM slot.\n\n"
    "Question: Which slot does each analyst have?"
)

SLOTS = ["9:00 AM", "11:00 AM", "2:00 PM"]
ANALYSTS = ["Priya", "Quinn", "Rae"]


def satisfies_all_clues(a: dict) -> bool:
    pos = {name: SLOTS.index(slot) for name, slot in a.items()}
    return (
        a["Priya"] != "9:00 AM"                                       # clue 1
        and pos["Quinn"] < pos["Rae"]                                  # clue 2
        and (a["Priya"] != "2:00 PM" or a["Quinn"] == "9:00 AM")       # clue 3
        and a["Rae"] != "11:00 AM"                                     # clue 4
    )


solutions = [dict(zip(ANALYSTS, p)) for p in permutations(SLOTS) if satisfies_all_clues(dict(zip(ANALYSTS, p)))]
assert len(solutions) == 1, "the puzzle must have exactly one solution"
SOLUTION = solutions[0]

print(TASK_INSTANCE)
print(f"\n✅ Brute force: exactly one valid assignment (kept hidden from the model)")

### 2.3 A Typed Answer

Every answer in this notebook - direct or Self-Discover - ends in the same `Assignment` object, so code can compare it with `SOLUTION`. `Literal` restricts each field to the three real slots.

In [ ]:
# ============================================================================
# ANSWER SCHEMA: One slot per analyst, restricted to the real slots
# ============================================================================
from typing import List, Literal

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

Slot = Literal["9:00 AM", "11:00 AM", "2:00 PM"]


class Assignment(BaseModel):
    Priya: Slot
    Quinn: Slot
    Rae: Slot


def is_correct(assignment: Assignment) -> bool:
    return assignment.model_dump() == SOLUTION


print(f"✅ Assignment fields: {list(Assignment.model_fields)}")

---
## 🧠 Part 3: SELECT → ADAPT → IMPLEMENT → SOLVE

### Stage 1 - SELECT

The LLM looks only at the **task type** (not the specific instance) and the catalog, and chooses the relevant modules.

In [ ]:
# ============================================================================
# STAGE 1 - SELECT: Choose relevant reasoning modules for this task type
# ============================================================================
class ModuleSelection(BaseModel):
    """The subset of the reasoning-module catalog relevant to this task type."""
    selected_modules: List[str] = Field(
        description="The exact module names (the text before the colon, e.g. 'Decomposition') "
        "copied verbatim from the provided catalog, for every module that is relevant to this task type."
    )
    rationale: str = Field(
        description="A brief explanation of why these modules were chosen and the others were not."
    )

select_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are the SELECT stage of a Self-Discover reasoning pipeline. You are given a catalog of "
        "generic reasoning modules and a description of a TASK TYPE (not a specific instance). "
        "Choose only the modules from the catalog that would genuinely help an agent reason through "
        "this type of task. Do not select modules that are irrelevant just to fill space, and do not "
        "invent new modules -- only choose from the catalog provided.",
    ),
    (
        "human",
        "Reasoning module catalog:\n{catalog}\n\nTask type:\n{task_type}\n\n"
        "Select the modules relevant to this task type.",
    ),
])

selector = select_prompt | llm.with_structured_output(ModuleSelection)

catalog_text = "\n".join(f"- {m}" for m in REASONING_MODULES)
selection = selector.invoke({"catalog": catalog_text, "task_type": TASK_TYPE_DESCRIPTION})

print("Selected modules:", selection.selected_modules)
print(f"Rationale: {selection.rationale}")

### Stage 2 - ADAPT

The selected modules are still generic ("Break the problem into sub-problems"). ADAPT rephrases each into an instruction specific to *this task type* - still without touching the specific instance.

In [ ]:
# ============================================================================
# STAGE 2 - ADAPT: Rephrase each module for this task type
# ============================================================================
class AdaptedModules(BaseModel):
    """Task-specific rephrasing of each selected generic reasoning module."""
    adapted_instructions: List[str] = Field(
        description="One task-specific instruction per selected module, in the same order as the "
        "selected modules, rephrasing each generic module in language specific to this task type."
    )

adapt_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are the ADAPT stage of a Self-Discover reasoning pipeline. You are given a TASK TYPE "
        "and a list of generic reasoning modules already selected as relevant. Rephrase each generic "
        "module into a concrete instruction specific to this task type -- still generic to the task "
        "TYPE, not to any one specific instance of it.",
    ),
    (
        "human",
        "Task type:\n{task_type}\n\nSelected generic modules:\n{modules}\n\n"
        "Adapt each module into a task-specific instruction, preserving order.",
    ),
])

adapter = adapt_prompt | llm.with_structured_output(AdaptedModules)

modules_text = "\n".join(f"- {m}" for m in selection.selected_modules)
adaptation = adapter.invoke({"task_type": TASK_TYPE_DESCRIPTION, "modules": modules_text})

for i, instr in enumerate(adaptation.adapted_instructions, 1):
    print(f"{i}. {instr}")

### Stage 3 - IMPLEMENT

The adapted instructions become one **ordered, structured reasoning plan** - the "discovered structure" for this task type. It is still not an answer: it is a reusable checklist that could be applied to any instance of the type.

In [ ]:
# ============================================================================
# STAGE 3 - IMPLEMENT: Compose an ordered reasoning structure
# ============================================================================
class ReasoningStep(BaseModel):
    step_number: int = Field(description="1-indexed position of this step in the reasoning plan.")
    instruction: str = Field(description="A single, concrete, actionable instruction for this step.")

class ReasoningStructure(BaseModel):
    """The reusable, discovered step-by-step reasoning plan for this task TYPE."""
    task_type: str = Field(description="A short label naming the general category of task this plan is for.")
    steps: List[ReasoningStep] = Field(description="The ordered sequence of steps making up the plan.")

implement_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are the IMPLEMENT stage of a Self-Discover reasoning pipeline. You are given a TASK "
        "TYPE and a set of task-specific adapted instructions. Compose them into ONE ordered, "
        "concrete, step-by-step reasoning plan (an algorithm) that an agent could follow to solve "
        "ANY specific instance of this task type. Do not solve any particular instance here -- "
        "produce only the reusable plan/structure, with each step phrased as an imperative "
        "instruction (e.g. 'List all entities and all possible slots.').",
    ),
    (
        "human",
        "Task type:\n{task_type}\n\nAdapted, task-specific instructions to compose:\n{instructions}\n\n"
        "Produce the ordered reasoning structure.",
    ),
])

implementer = implement_prompt | llm.with_structured_output(ReasoningStructure)

instructions_text = "\n".join(f"- {i}" for i in adaptation.adapted_instructions)
reasoning_structure = implementer.invoke({
    "task_type": TASK_TYPE_DESCRIPTION,
    "instructions": instructions_text,
})

structure_text = "\n".join(f"Step {s.step_number}: {s.instruction}" for s in reasoning_structure.steps)
print(f"Task type: {reasoning_structure.task_type}\n")
print(structure_text)

Notice that the structure says nothing about Priya, Quinn, Rae, or any specific time slot - it is generic to "multi-constraint assignment puzzles". **Stages 1-3 never look at the instance**, only at the shape of the problem. That is what makes the structure reusable, and inspectable before any effort is spent on a specific case.

### Stage 4 - SOLVE

Finally, the model gets **both** the discovered structure **and** the specific instance, and follows the structure step by step. The answer comes back as a worked trace plus a typed `Assignment`.

In [ ]:
# ============================================================================
# STAGE 4 - SOLVE: Follow the structure on the specific instance
# ============================================================================
class FinalAnswer(BaseModel):
    reasoning_trace: str = Field(
        description="The worked reasoning, following the discovered structure step by step, "
        "including any case analysis and the final verification pass against every clue."
    )
    assignment: Assignment = Field(description="The slot assigned to each analyst.")


solve_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are the SOLVE stage of a Self-Discover reasoning pipeline. You are given a discovered "
        "reasoning structure (a numbered checklist) and a specific task instance. Follow the "
        "structure step by step -- do not skip steps, especially any verification step -- and use it "
        "to solve the specific instance. Show your work in `reasoning_trace`, and give the concrete "
        "final assignment in `assignment`.",
    ),
    ("human", "Discovered reasoning structure:\n{structure}\n\nTask instance:\n{instance}"),
])

solver = solve_prompt | sampling_llm.with_structured_output(FinalAnswer)
solution = solver.invoke({"structure": structure_text, "instance": TASK_INSTANCE})

print("--- Reasoning trace (first 1,200 characters) ---")
print(solution.reasoning_trace[:1200])
print(f"\n--- Final answer --- {solution.assignment.model_dump()}  {'✅ correct' if is_correct(solution.assignment) else '❌ wrong'}")

---
## 📏 Part 4: Measuring the Difference

A single answer proves little either way, so sample several. The **direct** baseline asks for the assignment with no reasoning at all ("final answer only"); the **Self-Discover** answers reuse the structure discovered above - SELECT/ADAPT/IMPLEMENT ran once, only SOLVE runs per sample. Both use the same model at temperature 0.7, and code checks every answer against the brute-force solution.

In [ ]:
# ============================================================================
# SAMPLE BOTH APPROACHES: 5 direct answers vs. 3 Self-Discover answers
# ============================================================================
direct_prompt = ChatPromptTemplate.from_messages([
    ("system", "Solve the puzzle. Give the final answer only."),
    ("human", "{task}"),
])
direct_chain = direct_prompt | sampling_llm.with_structured_output(Assignment)

direct_answers = direct_chain.batch([{"task": TASK_INSTANCE}] * 5)
discover_answers = [a.assignment for a in solver.batch([{"structure": structure_text, "instance": TASK_INSTANCE}] * 3)]

for label, answers in [("Direct", direct_answers), ("Self-Discover", discover_answers)]:
    correct = sum(is_correct(a) for a in answers)
    print(f"{label:>14}: {correct}/{len(answers)} correct")
    for a in answers:
        print(f"{'':>16}{'✅' if is_correct(a) else '❌'} {a.model_dump()}")

Read the counts. In our runs the direct answers were rarely right - between 0 of 5 and 2 of 6 - and usually wrong in the same way: Rae at 11:00 AM, which clue 4 forbids. The Self-Discover answers were right far more often - 3 of 3, 4 of 4 and 2 of 3 in our runs - though not always. The structure did not add knowledge; it made the steps that catch those mistakes (enumerate cases, check every clue against the final assignment) **mandatory** instead of optional.

The gain has limits. On a harder five-person version of this puzzle, `gpt-4o-mini` failed with and without the discovered structure: a checklist helps a model apply reasoning it can do, not reasoning beyond its reach.

---
## 📝 Summary

In this notebook, we learned:

### 1. The Self-Discover pattern
- SELECT → ADAPT → IMPLEMENT only look at the task *type*; SOLVE is the only stage that sees the instance.

### 2. SELECT, ADAPT, IMPLEMENT
- A fixed catalog plus typed outputs gives a reasoning plan you can print, inspect and reuse across instances of the same type.

### 3. SOLVE
- Following the plan makes verification steps mandatory - the model must check every clue before answering.

### 4. Measuring the difference
- Brute-force ground truth plus sampled answers turns "more likely to get it right" into a measured rate. Structure helps within the model's reach, not beyond it.

### Next Steps
- **`06_constitutional_ai_critique_loop.ipynb`**: critique and revise a draft against a written list of named rules